# 02 — Case study: the four skills, evaluated end to end

Each of the four skills is run through its contract and scored against the
baseline and metric fixed in `docs/requirement.md`. This notebook is the
narrated version of `python -m src.run_all`; the artifacts it writes are what
the README reports.

> Models download from the Hub on first run, then cache. The Triage eval is
> capped (`n`) so the notebook stays interactive.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
from src import evaluate

## Skill 1 — Triage (sentiment)

Transformer vs VADER vs majority-class, on the rating-proxy labels. Acceptance
bar: **beat VADER on macro-F1**.

In [ ]:
triage = evaluate.evaluate_triage(n=300)
print(json.dumps(triage, indent=2))

## Skill 2 — Translate (EN -> ES)

BLEU + chrF on the small reference set. BLEU is brittle on short text, so chrF
rides alongside; both are indicative, not a benchmark.

In [ ]:
translate = evaluate.evaluate_translate()
print(json.dumps(translate, indent=2, ensure_ascii=False))

## Skill 3 — Answer (extractive QA)

Exact-match / token-F1 on the small hand-labeled set. Illustrative that the
skill is wired correctly.

In [ ]:
answer = evaluate.evaluate_answer()
print(json.dumps(answer, indent=2))

## Skill 4 — Digest (summarization)

ROUGE vs the lead-3 baseline. Acceptance bar: **beat lead-3 on ROUGE-L, or
ship lead-3 and say so.**

In [ ]:
digest = evaluate.evaluate_digest()
print(json.dumps(digest, indent=2))

## The assistant ties them together

The routing agent turns a free-text message into the right skill call. With
Ollama running it uses the LLM router; offline it falls back to keyword
routing (shown here).

In [ ]:
from src.agent import chat
out = chat("Is this review positive? 'Best truck I have ever owned, no regrets.'", use_ollama=False)
print(out['via'], '->', out['skill'])
print(out['reply'])